[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vladimiracunadev-create/python-data-science-bootcamp/blob/master/classes/10-modelos-supervisados/soluciones.ipynb)

# SOLUCIONES — Clase 10: Modelos Supervisados: Clasificacion

> **Instrucciones:** Este archivo contiene las soluciones completas a todos los ejercicios. Intenta resolver el `notebook.ipynb` antes de consultar estas soluciones.

---

## Objetivos de la clase

- **Entender** la diferencia entre clasificacion y regresion
- **Entrenar** un `DecisionTreeClassifier` paso a paso
- **Interpretar** la matriz de confusion y sus cuatro cuadrantes
- **Comparar** el arbol de decision con `LogisticRegression`


## Concepto clave: Regresion vs Clasificacion

```
REGRESION vs CLASIFICACION
==========================

Regresion: predice un NUMERO CONTINUO
  Input: [antiguedad=24, productos=3]  ->  Output: 45200 (ventas predichas)

Clasificacion: predice una CATEGORIA
  Input: [antiguedad=24, productos=3]  ->  Output: CHURN=1 (cliente se va)
                                                   CHURN=0 (cliente se queda)
```


In [ ]:
# ============================================================
# BLOQUE: Importacion de librerias
# ============================================================

import pandas as pd          # tablas de datos (DataFrames)
import numpy as np           # operaciones matematicas vectorizadas
import matplotlib.pyplot as plt  # graficos paso a paso

from sklearn.tree import DecisionTreeClassifier      # arbol de decision
from sklearn.linear_model import LogisticRegression  # regresion logistica

from sklearn.model_selection import train_test_split  # dividir datos en train/test
from sklearn.model_selection import cross_val_score   # validacion cruzada K-Fold

from sklearn.metrics import confusion_matrix          # matriz 2x2 de errores
from sklearn.metrics import ConfusionMatrixDisplay    # grafico de la matriz
from sklearn.metrics import classification_report     # precision, recall, f1 por clase
from sklearn.metrics import f1_score                  # F1 individual

from sklearn.preprocessing import StandardScaler  # escalar a media=0, std=1

print("Librerias cargadas correctamente")


## Seccion 1: Cargar y explorar `retencion_clientes.csv`


In [ ]:
# ============================================================
# SOLUCION: Carga y exploracion del dataset
# ============================================================

# Paso 1: leer el archivo CSV en un DataFrame
df = pd.read_csv("../../datasets/retencion_clientes.csv")  # leer CSV desde ruta relativa

# Paso 2: ver dimensiones
print(f"Dimensiones: {df.shape}")
print()

# Paso 3: ver primeras 3 filas
print("Primeras 3 filas:")
print(df.head(3))
print()

# Paso 4: conteo de cada clase del target
print("Distribucion de churned:")
print(df["churned"].value_counts())
print()

# Paso 5: porcentaje de cada clase
pct = df["churned"].value_counts(normalize=True) * 100  # proporciones en %
print("Porcentaje por clase:")
print(pct.round(1))


## Estructura del dataset

```
DATASET: retencion_clientes.csv
================================

| antiguedad_meses | n_productos | reclamos_ultimo_ano | segmento | churned |
        ^               ^                 ^                ^           ^
   FEATURES (X): variables predictoras                      TARGET (y)
                                                        0 = se quedo
                                                        1 = se fue (churn)
```


In [ ]:
# ============================================================
# SOLUCION: Seleccion de features y target
# ============================================================

# Identificar columnas numericas
cols_num = df.select_dtypes(include=["number"]).columns.tolist()  # solo int64 y float64
print(f"Columnas numericas: {cols_num}")

# Excluir el target de la lista de features
features = [c for c in cols_num if c != "churned"]  # features sin el target
print(f"Features seleccionadas: {features}")

# Crear X (matriz de features) e y (vector target)
X = df[features]       # DataFrame (n_muestras, n_features)
y = df["churned"]      # Serie con valores 0 o 1

print(f"\nShape X: {X.shape}")
print(f"Shape y: {y.shape}")


In [ ]:
# ============================================================
# SOLUCION: Division Train / Test
# ============================================================

# Dividir con stratify para mantener el balance de clases en ambas particiones
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,   # 20% para test
    random_state=42, # reproducibilidad
    stratify=y       # mismo % de churn en train y test
)

print(f"Entrenamiento: {X_train.shape[0]} filas")
print(f"Prueba:        {X_test.shape[0]} filas")

# Verificar balance de clases
print("\nBalance en TRAIN:")
print((y_train.value_counts(normalize=True) * 100).round(1))
print("\nBalance en TEST:")
print((y_test.value_counts(normalize=True) * 100).round(1))


## Como funciona un Arbol de Decision

```
ARBOL DE DECISION — Como funciona
===================================

         Pregunta raiz: antiguedad_meses > 24?
                       /                    \
                     SI                      NO
              reclamos > 2?          n_productos > 1?
              /         \              /             \
          CHURN=1    CHURN=0       CHURN=1         CHURN=0

max_depth controla la profundidad. Mas profundidad = riesgo de overfitting.
```


In [ ]:
# ============================================================
# SOLUCION: Entrenamiento del Arbol de Decision
# ============================================================

# Paso 1: crear el clasificador con hiperparametros controlados
arbol = DecisionTreeClassifier(
    max_depth=4,          # profundidad maxima para generalizar bien
    min_samples_leaf=5,   # minimo 5 muestras por hoja para evitar overfitting
    random_state=42       # reproducibilidad del resultado
)

# Paso 2: entrenar el modelo
arbol.fit(X_train, y_train)  # aprender reglas de decision sobre los datos de train

# Paso 3: predecir sobre el test
y_pred_arbol = arbol.predict(X_test)  # aplicar las reglas a datos nuevos

# Paso 4: calcular exactitud
exactitud = arbol.score(X_test, y_test)  # % de predicciones correctas
print(f"Exactitud del Arbol de Decision: {exactitud:.4f} ({exactitud*100:.1f}%)")


## Seccion 2: Matriz de Confusion

```
MATRIZ DE CONFUSION
====================

                  PREDICHO: NO churn   PREDICHO: SI churn
REAL: NO churn  [  TN (correcto)     |  FP (falsa alarma)  ]
REAL: SI churn  [  FN (error grave)  |  TP (correcto)      ]

FN = ERROR GRAVE: cliente que se va y no detectamos
```


In [ ]:
# ============================================================
# SOLUCION: Visualizacion de la Matriz de Confusion
# ============================================================

# Calcular la matriz numerica [[TN, FP], [FN, TP]]
cm = confusion_matrix(y_test, y_pred_arbol)  # comparar reales vs predicciones
print("Matriz de confusion:")
print(cm)
print()

# Extraer los cuatro cuadrantes
tn, fp, fn, tp = cm.ravel()  # aplanar 2x2 a 4 valores
print(f"  TN={tn}  FP={fp}  FN={fn}  TP={tp}")
print(f"  FN (error critico para negocio): {fn}")
print()

# Visualizar como heatmap
fig, ax = plt.subplots(figsize=(6, 5))  # lienzo de 6x5 pulgadas
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["NO churn (0)", "SI churn (1)"]
)
disp.plot(ax=ax, cmap="Blues", colorbar=False)  # heatmap azul sin barra lateral
ax.set_title("Matriz de Confusion — Arbol de Decision", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# SOLUCION: Reporte de Clasificacion
# ============================================================

# classification_report muestra precision, recall, f1 y support por clase
# precision = TP/(TP+FP)  — calidad de las predicciones positivas
# recall    = TP/(TP+FN)  — cobertura sobre los positivos reales
# f1        = 2*p*r/(p+r) — media armonica de precision y recall
# support   = conteo real de cada clase en el test set

print("Reporte de Clasificacion — Arbol de Decision:")
print("=" * 55)
print(
    classification_report(
        y_test,
        y_pred_arbol,
        target_names=["NO churn", "SI churn"]  # nombres legibles
    )
)


## Seccion 3: Regresion Logistica con Sigmoide

```
FUNCION SIGMOIDE — Convierte puntajes lineales en probabilidades
================================================================

  P(churn=1) = 1 / (1 + e^(-(w*X + b)))

  1.0 |                              ______
      |                         ____/
  0.5 |------------------------/  <- umbral (P > 0.5 -> clase 1)
      |                   ____/
  0.0 |__________________/

Requiere features en la misma escala -> StandardScaler
```


In [ ]:
# ============================================================
# SOLUCION: Regresion Logistica con StandardScaler
# ============================================================

# Paso 1: crear el escalador
scaler = StandardScaler()  # aprende media y std de cada columna

# Paso 2: fit+transform solo en train (NUNCA fit en test)
X_train_sc = scaler.fit_transform(X_train)  # aprende escala del train y transforma

# Paso 3: solo transform en test (usando la escala del train)
X_test_sc = scaler.transform(X_test)  # aplica la misma escala del train al test

# Paso 4: entrenar la Regresion Logistica con datos escalados
rl = LogisticRegression(
    max_iter=1000,  # suficientes iteraciones para convergencia
    random_state=42 # reproducibilidad
)
rl.fit(X_train_sc, y_train)  # entrenar con features escaladas

# Paso 5: predecir
y_pred_rl = rl.predict(X_test_sc)  # clases predichas: 0 o 1

# Paso 6: reporte completo
print("Reporte de Clasificacion — Regresion Logistica:")
print("=" * 55)
print(
    classification_report(
        y_test, y_pred_rl,
        target_names=["NO churn", "SI churn"]
    )
)


In [ ]:
# ============================================================
# SOLUCION: Comparacion de modelos
# ============================================================

# Calcular accuracy y F1 weighted para cada modelo
acc_arbol = arbol.score(X_test, y_test)                              # accuracy del arbol
acc_rl    = rl.score(X_test_sc, y_test)                              # accuracy de RL
f1_arbol  = f1_score(y_test, y_pred_arbol, average="weighted")       # F1 del arbol
f1_rl     = f1_score(y_test, y_pred_rl,    average="weighted")       # F1 de RL

# Crear tabla comparativa
resultados = {
    "Arbol de Decision":   {"Accuracy": round(acc_arbol, 4), "F1 (weighted)": round(f1_arbol, 4)},
    "Regresion Logistica": {"Accuracy": round(acc_rl,    4), "F1 (weighted)": round(f1_rl,    4)},
}
df_comp = pd.DataFrame.from_dict(resultados, orient="index")  # tabla con modelos como filas

print("Comparacion de modelos:")
print("=" * 45)
print(df_comp.to_string())
print()

mejor = df_comp["F1 (weighted)"].idxmax()  # modelo con mayor F1
print(f"Modelo recomendado: {mejor}")


## SOLUCION al Ejercicio: Agregar `reclamos_ultimo_ano` como feature


In [ ]:
# ============================================================
# SOLUCION EJERCICIO: Re-entrenar con 'reclamos_ultimo_ano'
# ============================================================

# Paso 1: redefinir features incluyendo la nueva columna
# Verificamos que 'reclamos_ultimo_ano' existe en el DataFrame
if "reclamos_ultimo_ano" in df.columns:  # comprobar existencia de la columna
    features_v2 = features + ["reclamos_ultimo_ano"]  # agregar nueva feature a la lista
else:
    # Si la columna ya estaba incluida en 'features', no hay que agregarla
    print("'reclamos_ultimo_ano' ya estaba en features o no existe. Usando features actuales.")
    features_v2 = features

print(f"Nuevas features: {features_v2}")

# Paso 2: re-crear X con la nueva lista de features
X_v2 = df[features_v2]  # matriz de features actualizada

# Paso 3: re-dividir datos (mismo random_state para comparacion justa)
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X_v2, y, test_size=0.2, random_state=42, stratify=y
)

# Paso 4: re-entrenar el Arbol de Decision
arbol_v2 = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)
arbol_v2.fit(X_tr2, y_tr2)  # entrenar con el nuevo conjunto de features
y_pred_arbol_v2 = arbol_v2.predict(X_te2)  # predecir

# Paso 5: re-escalar y re-entrenar Regresion Logistica
scaler_v2 = StandardScaler()  # nuevo escalador para las nuevas features
X_tr2_sc = scaler_v2.fit_transform(X_tr2)  # ajustar y transformar train
X_te2_sc = scaler_v2.transform(X_te2)      # transformar test con escala del train
rl_v2 = LogisticRegression(max_iter=1000, random_state=42)
rl_v2.fit(X_tr2_sc, y_tr2)  # entrenar RL con datos escalados
y_pred_rl_v2 = rl_v2.predict(X_te2_sc)  # predecir

# Paso 6: comparar F1 antes y despues
f1_arbol_v2 = f1_score(y_te2, y_pred_arbol_v2, average="weighted")
f1_rl_v2    = f1_score(y_te2, y_pred_rl_v2,    average="weighted")

print("\nComparacion ANTES vs DESPUES de agregar reclamos_ultimo_ano:")
print("=" * 60)
print(f"  Arbol v1 F1: {f1_arbol:.4f}  |  Arbol v2 F1: {f1_arbol_v2:.4f}  Delta: {f1_arbol_v2-f1_arbol:+.4f}")
print(f"  RL    v1 F1: {f1_rl:.4f}    |  RL    v2 F1: {f1_rl_v2:.4f}    Delta: {f1_rl_v2-f1_rl:+.4f}")
print()
mejora_arbol = "MEJORA" if f1_arbol_v2 > f1_arbol else "NO mejora"
mejora_rl    = "MEJORA" if f1_rl_v2 > f1_rl    else "NO mejora"
print(f"  Arbol de Decision: {mejora_arbol}")
print(f"  Regresion Logistica: {mejora_rl}")


## Extra: Feature Importances del Arbol de Decision

Una ventaja clave del Arbol de Decision es su **interpretabilidad**: podemos ver exactamente que features usa mas para tomar decisiones.


In [ ]:
# ============================================================
# EXTRA: Feature Importances del DecisionTree
# ============================================================

# .feature_importances_ es un array numpy con la importancia de cada feature
# Los valores estan entre 0 y 1 y suman exactamente 1.0
# Representa la reduccion total de impureza de Gini atribuida a cada feature
importancias = arbol_v2.feature_importances_  # importancia por feature

# Paso 1: crear DataFrame para facilitar la visualizacion
# Asociar cada importancia con el nombre de su columna correspondiente
df_imp = pd.DataFrame({
    "feature":     list(X_v2.columns),  # nombres de las features
    "importancia": importancias          # valores de importancia
})

# Paso 2: ordenar de mayor a menor importancia
df_imp = df_imp.sort_values("importancia", ascending=False).reset_index(drop=True)

# Paso 3: mostrar la tabla de importancias
print("Feature Importances del Arbol de Decision (v2):")
print("=" * 45)
print(df_imp.to_string(index=False))  # tabla sin indice numerico
print(f"\nSuma de importancias: {importancias.sum():.6f} (debe ser 1.0)")

# Paso 4: grafico de barras horizontal
fig, ax = plt.subplots(figsize=(8, max(3, len(df_imp) * 0.5)))  # alto dinamico

# barh: barras horizontales (mejor para leer nombres de features)
ax.barh(
    df_imp["feature"],          # eje y: nombres de features
    df_imp["importancia"],      # eje x: valores de importancia
    color="steelblue",          # color de las barras
    edgecolor="white",          # borde blanco para separacion visual
    height=0.6                  # grosor de cada barra
)

# Agregar etiquetas con el valor exacto al lado de cada barra
for i, val in enumerate(df_imp["importancia"]):
    ax.text(
        val + 0.005,   # posicion x: ligeramente a la derecha de la barra
        i,             # posicion y: en el centro de la barra
        f"{val:.3f}",  # formato con 3 decimales
        va="center",   # alineacion vertical centrada
        fontsize=10    # tamano de la etiqueta
    )

ax.set_xlabel("Importancia (reduccion de impureza Gini)", fontsize=12)
ax.set_title("Feature Importances — Arbol de Decision (v2)", fontsize=14, fontweight="bold")
ax.grid(True, alpha=0.3, axis="x")  # cuadricula solo en eje x
ax.invert_yaxis()  # poner la feature mas importante arriba

plt.tight_layout()
plt.show()


## Resumen de la Clase 10 — Checklist completado

- [x] Diferencia entre regresion (numero continuo) y clasificacion (categoria discreta)
- [x] Como preparar features (X) y target (y) para clasificacion
- [x] `train_test_split` con `test_size=0.2`, `random_state=42`, `stratify=y`
- [x] Entrenamiento de `DecisionTreeClassifier` con `max_depth=4`, `min_samples_leaf=5`
- [x] Lectura de la **matriz de confusion**: TN, TP, FP, FN
- [x] `classification_report`: precision, recall, f1-score, support
- [x] `StandardScaler` antes de `LogisticRegression` (evitar dominancia de escala)
- [x] Comparacion de modelos en tabla (accuracy y F1 weighted)
- [x] Ejercicio: agregar feature y comparar mejora de F1
- [x] Extra: `feature_importances_` del arbol de decision

**Proxima clase:** Evaluacion robusta con Cross-Validation y Pipelines de ML.
